# Torres de Hanoi — Diseño en Arte ASCII

**Curso:** Programación Avanzada  
**Kernel:** xeus-cling (C++17)

---

En este cuaderno vamos a construir paso a paso la base visual para las **Torres de Hanoi en arte ASCII**.

- **Arte ASCII**: dibujar imágenes usando solo caracteres de texto
- **Representación del estado**: cómo guardar en memoria qué discos tiene cada torre
- **Funciones de dibujo**: traducir ese estado en una imagen en la consola
- **Funciones de movimiento**: mover un disco de una torre a otra con validación

El resultado de hoy es el **punto de partida de la actividad en equipo**: tú y tu equipo van a tomar esta base y crear su propio diseño visual, tan creativo como quieran.

---

## Requisitos previos

In [ ]:
#include <iostream>
#include <vector>
#include <string>
using namespace std;

---

## Parte 1 — ¿Qué es el arte ASCII?

El **arte ASCII** es la práctica de crear imágenes, diagramas o animaciones usando únicamente los 95 caracteres imprimibles del estándar ASCII: letras, números y símbolos como `|`, `=`, `*`, `#`, `/`, `\`, etc.

Nació en los años 60 cuando las impresoras y terminales no podían mostrar gráficos — solo texto. Los programadores descubrieron que combinando caracteres podían crear figuras sorprendentemente expresivas.

### Algunos ejemplos rápidos

```
Gato simple:               Robot:                   Cohete:
  /\_/\                    [o_o]                       /\
 ( o.o )                   |---|                      /  \
  > ^ <                    |   |                     | () |
                           /___\                    |    |
                                                      \  /
                                                      /  \
                                                     /    \
                                                    /______\
                                                       ||
```

### ¿Por qué sigue siendo relevante?

- Interfaces de **terminal y consola** (herramientas como `htop`, `git log --graph`)
- Diagramas en **comentarios de código** y documentación
- **Juegos de texto** clásicos (Rogue, NetHack — activos desde los 80 hasta hoy)
- Arte generativo en plataformas como Twitter/X
- **Diagramas de arquitectura** en archivos de texto plano (como los del cuaderno de apuntadores)

### Recursos para inspirarte

| Galería | Qué encontrarás |
|---|---|
| [asciiart.eu](https://www.asciiart.eu/) | Biblioteca enorme categorizada: animales, películas, logos |
| [ascii.co.uk/art](https://ascii.co.uk/art) | Colección clásica con ejemplos históricos |
| [textfancy.com](https://textfancy.com/) | Generador de texto decorativo con caracteres Unicode |
| [patorjk.com/software/taag](https://patorjk.com/software/taag/) | **Figlet online**: convierte texto en arte ASCII con docenas de fuentes |
| [ascii-generator.site](https://ascii-generator.site/) | Convierte imágenes a arte ASCII automáticamente |

### Ejemplo: el nombre del curso en arte ASCII (fuente "Big")

```
 ____                                                  _               _           
|  _ \ _ __ ___   __ _ _ __ __ _ _ __ ___   __ _  ___(_) ___  _ __   / \__   ___  ___ 
| |_) | '__/ _ \ / _` | '__/ _` | '_ ` _ \ / _` |/ __| |/ _ \| '_ \ / _ \ \ / / |/ _ \/ __|
|  __/| | | (_) | (_| | | | (_| | | | | | | (_| | (__| | (_) | | | / ___ \ V /| | (_) \__ \
|_|   |_|  \___/ \__, |_|  \__,_|_| |_| |_|\__,_|\___|_|\___/|_| |_/_/   \_\_/ |_|\___/|___/
                  |___/                                                                        
```

---

## Parte 2 — El punto de partida: torres vacías

El material inicial de la sesión nos da este código como base.  
Vamos a ejecutarlo, entender cada función, y luego mejorarlo.

In [ ]:
// ── Código base de la sesión ────────────────────────────────────────────────
void pintarTorre()  { cout << "    |    "; }
void pintarVacio()  { for(int i = 0; i < 3; i++) pintarTorre(); }
void pintarBase()   { cout << "============================="; }

void pintarTorres() {
    for(int i = 0; i < 6; i++) {
        pintarVacio();
        cout << endl;
    }
    pintarBase();
    cout << endl;
}

pintarTorres();

    |        |        |    
    |        |        |    
    |        |        |    
    |        |        |    
    |        |        |    
    |        |        |    


**¿Qué hace cada función?**

| Función | Qué imprime | Ancho |
|---|---|---|
| `pintarTorre()` | `    \|    ` — un poste centrado | 9 caracteres |
| `pintarVacio()` | tres postes seguidos — una fila vacía | 27 caracteres |
| `pintarBase()` | una línea de `=` | 29 caracteres |
| `pintarTorres()` | 6 filas vacías + base | imagen completa |

Hay un detalle: los postes miden 27 caracteres en total pero la base mide 29.  
Eso es algo que **tú y tu equipo pueden ajustar** al diseñar su versión.

### La pregunta que guía el resto del cuaderno

> *Si los postes están vacíos ahora, ¿cómo dibujamos discos en ellos?*

---

## Parte 3 — Representar discos con ASCII

Cada disco tiene un **número** (su tamaño).  
El disco 1 es el más pequeño, el disco `N` es el más grande.  
En la pantalla, un disco de tamaño `k` se dibuja como una barra de `2k - 1` caracteres `=`, centrada en el poste.

Para `N = 4` discos, así se verían las filas de cada tamaño, en un ancho de torre de 9 caracteres:

```
  vacío:    "    |    "   →   solo el poste
  disco 1:  "   ===   "   →   3 chars (2×1−1),  3 espacios a cada lado
  disco 2:  "  =====  "   →   5 chars (2×2−1),  2 espacios a cada lado
  disco 3:  " ======= "   →   7 chars (2×3−1),  1 espacio a cada lado
  disco 4:  "========="   →   9 chars (2×4−1),  0 espacios
```

La fórmula para los espacios en un poste de ancho `W` con un disco de tamaño `k`:
- ancho del disco: `2*k - 1`
- espacios a cada lado: `(W - (2*k - 1)) / 2`

In [ ]:
// Dibuja una sola "celda" de un poste:
//   k == 0  → fila vacía (solo el poste)
//   k >= 1  → fila con un disco de tamaño k
void dibujarFila(int k, int anchoTorre = 9) {
    if (k == 0) {
        // celda vacía: espacios + poste + espacios
        int mitad = anchoTorre / 2;
        cout << string(mitad, ' ') << '|' << string(mitad, ' ');
    } else {
        int anchoDisco = 2 * k - 1;
        int espacios   = (anchoTorre - anchoDisco) / 2;
        cout << string(espacios, ' ')
             << string(anchoDisco, '=')
             << string(espacios, ' ');
    }
}

// Probamos todas las filas posibles para N=4
int N = 4;
cout << "Así se ven las filas de cada tamaño (torre de 9 chars):" << endl;
cout << "|"; dibujarFila(0); cout << "|  ← vacío"   << endl;
for (int k = 1; k <= N; k++) {
    cout << "|"; dibujarFila(k); cout << "|  ← disco " << k << endl;
}

Así se ven las filas de cada tamaño (torre de 9 chars):
|    |    |  ← vacío
|   ===   |  ← disco 1
|  =====  |  ← disco 2
| ======= |  ← disco 3
|=========|  ← disco 4


---

## Parte 4 — El estado: ¿qué hay en cada torre?

Para dibujar las torres necesitamos saber **qué discos tiene cada torre y en qué orden**.  
Usamos un `vector<int>` por torre: el índice 0 es el fondo, el último índice es el tope.

```
Estado inicial (todos los discos en la torre A):

  torre A = {4, 3, 2, 1}   ← disco 4 al fondo, disco 1 arriba
  torre B = {}
  torre C = {}

Visualmente:
       |              |              |
      ===             |              |
     =====            |              |
    =======           |              |
   =========          |              |
  ═══════════════  ═══════════════  ═══════════════
       A                 B                 C
```

In [ ]:
// Estado de las 3 torres como vectores
// Índice 0 = fondo de la torre, último índice = tope
vector<int> torreA = {4, 3, 2, 1};  // todos los discos al inicio
vector<int> torreB = {};
vector<int> torreC = {};

// Para ver el contenido:
auto imprimirEstado = [](const vector<int>& t, const string& nombre) {
    cout << nombre << ": [";
    for (int i = 0; i < (int)t.size(); i++) {
        cout << t[i];
        if (i + 1 < (int)t.size()) cout << ", ";
    }
    cout << "]" << endl;
};

imprimirEstado(torreA, "A");
imprimirEstado(torreB, "B");
imprimirEstado(torreC, "C");

A: [4, 3, 2, 1]
B: []
C: []


---

## Parte 5 — Dibujar las tres torres con su estado

Ahora unimos las piezas: dado el estado de las tres torres, dibujamos la imagen.

**La idea:** para cada fila (de arriba hacia abajo), miramos qué hay en esa posición de cada torre.

```
Con N = 4 discos, hay 4 filas de disco + 1 fila base.
La fila más alta (fila 0) muestra el tope; la más baja (fila N-1) muestra el fondo.

Fila i de la torre T:
  → si T tiene un disco en la posición i (contando desde arriba), dibujamos ese disco
  → si no, dibujamos vacío
```

La torre A con `{4, 3, 2, 1}` vista desde el índice:
- fondo (índice 0) = disco 4 → aparece en la **fila más baja**
- tope (índice 3) = disco 1 → aparece en la **fila más alta con disco**

In [ ]:
// Dibuja las 3 torres con su estado actual
// N_FILAS: cuántas filas de disco muestra (altura de la torre, sin contar base)
void dibujarTorres(const vector<int>& A, const vector<int>& B,
                   const vector<int>& C, int nFilas = 5) {
    int anchoTorre = 9;
    string separador = "  ";  // espacio entre torres

    // Dibujar fila por fila de arriba hacia abajo
    for (int fila = nFilas - 1; fila >= 0; fila--) {
        // Para cada torre: ¿hay un disco en esta fila?
        // La fila 0 (abajo) corresponde al fondo del vector
        for (const auto& torre : {A, B, C}) {
            int disco = 0;  // 0 = vacío
            if (fila < (int)torre.size()) {
                disco = torre[fila];  // el disco en esa posición
            }
            dibujarFila(disco, anchoTorre);
            cout << separador;
        }
        cout << endl;
    }

    // Base
    string base = string(anchoTorre, '=');
    cout << base << separador << base << separador << base << endl;
    cout << "    A       " << "    B       " << "    C    " << endl;
}

// --- Probamos con el estado inicial ---
cout << "Estado inicial:" << endl;
dibujarTorres(torreA, torreB, torreC);

Estado inicial:
    |          |          |    
   ===         |          |    
  =====        |          |    
 =======       |          |    
=========      |          |    
=========  =========  =========
    A           B           C    


**¿Por qué iteramos `fila` de `nFilas-1` hasta `0`?**  
Porque queremos dibujar de arriba hacia abajo en la pantalla.  
El índice 0 del vector es el fondo de la torre (aparece en la pantalla en la fila más baja), por eso invertimos el recorrido.

---

## Parte 6 — Mover un disco: validación y ejecución

En el juego de Hanoi hay una **regla de oro**: nunca puedes colocar un disco grande sobre uno pequeño.

Antes de mover, hay que validar:
1. La torre de **origen** no debe estar vacía.
2. La torre de **destino** debe estar vacía **o** el disco de su tope debe ser más grande que el disco que vamos a mover.

In [ ]:
// Intenta mover el disco del tope de 'origen' al tope de 'destino'.
// Retorna true si el movimiento fue válido, false si no.
bool moverDisco(vector<int>& origen, vector<int>& destino,
                const string& nomOrigen, const string& nomDestino) {
    // Validación 1: origen no vacío
    if (origen.empty()) {
        cout << "✗ La torre " << nomOrigen << " está vacía." << endl;
        return false;
    }

    int disco = origen.back();  // el disco en el tope

    // Validación 2: destino vacío o disco destino más grande
    if (!destino.empty() && destino.back() < disco) {
        cout << "✗ No puedes poner el disco " << disco
             << " sobre el disco " << destino.back() << "." << endl;
        return false;
    }

    // Movimiento válido
    origen.pop_back();
    destino.push_back(disco);
    cout << "✓ Disco " << disco << " movido de " << nomOrigen
         << " a " << nomDestino << "." << endl;
    return true;
}

// --- Probamos algunos movimientos ---
cout << endl << "Movimiento 1 — A → C:" << endl;
moverDisco(torreA, torreC, "A", "C");
dibujarTorres(torreA, torreB, torreC);


Movimiento 1 — A → C:
✓ Disco 1 movido de A a C.
    |          |          |    
    |          |          |    
  =====        |          |    
 =======       |          |    
=========      |         ===   
=========  =========  =========
    A           B           C    


In [ ]:
cout << "Movimiento 2 — A → B:" << endl;
moverDisco(torreA, torreB, "A", "B");
dibujarTorres(torreA, torreB, torreC);

cout << endl << "Movimiento inválido — C → B (disco 1 sobre disco 2):" << endl;
moverDisco(torreC, torreB, "C", "B");

Movimiento 2 — A → B:
✓ Disco 2 movido de A a B.
    |          |          |    
    |          |          |    
    |          |          |    
 =======      ===         |    
=========   =====        ===   
=========  =========  =========
    A           B           C    

Movimiento inválido — C → B (disco 1 sobre disco 2):
✗ No puedes poner el disco 1 sobre el disco 2.


---

## Parte 7 — Interfaz de usuario simple

Una interfaz mínima para consola: el usuario escribe la torre de origen y destino, el programa dibuja el estado actualizado.

En el cuaderno Jupyter `cin` no funciona de forma interactiva, así que simulamos la entrada con una lista de movimientos predefinidos.

In [ ]:
// Función auxiliar: convierte 'A', 'B' o 'C' en una referencia al vector
// (esto es solo para simplificar la interfaz; en consola se haría con cin)
auto& seleccionarTorre(char letra,
                       vector<int>& A, vector<int>& B, vector<int>& C) {
    if (letra == 'A') return A;
    if (letra == 'B') return B;
    return C;
}

// Simulación de una partida completa con movimientos fijos
{
    vector<int> A = {3, 2, 1};
    vector<int> B = {};
    vector<int> C = {};

    // Secuencia óptima para 3 discos: 7 movimientos
    // (no pienses aún por qué — eso es la sesión 13)
    vector<pair<char,char>> jugadas = {
        {'A','C'}, {'A','B'}, {'C','B'},
        {'A','C'}, {'B','A'}, {'B','C'}, {'A','C'}
    };

    cout << "== Estado inicial ==" << endl;
    dibujarTorres(A, B, C, 4);

    for (int i = 0; i < (int)jugadas.size(); i++) {
        auto [orig, dest] = jugadas[i];
        cout << endl << "== Movimiento " << i+1
             << ": " << orig << " → " << dest << " ==" << endl;
        moverDisco(seleccionarTorre(orig, A, B, C),
                   seleccionarTorre(dest, A, B, C),
                   string(1,orig), string(1,dest));
        dibujarTorres(A, B, C, 4);
    }
}

== Estado inicial ==
    |          |          |    
   ===         |          |    
  =====        |          |    
 =======       |          |    
=======    =======    =======
    A           B           C    

== Movimiento 1: A → C ==
✓ Disco 1 movido de A a C.
    |          |          |    
    |          |          |    
  =====        |          |    
 =======       |         ===   
=======    =======    =======
    A           B           C    

== Movimiento 2: A → B ==
✓ Disco 2 movido de A a B.
    |          |          |    
    |          |          |    
    |          |          |    
 =======      ===        ===   
=======    =======    =======
    A           B           C    

== Movimiento 3: C → B ==
✓ Disco 1 movido de C a B.
    |          |          |    
    |          |          |    
    |         ===         |    
 =======     =====        |    
=======    =======    =======
    A           B           C    

== Movimiento 4: A → C ==
✓ Disco 3 movido de A 

---

## Parte 8 — Actividad del equipo

Ahora es tu turno.  Trabaja con tu equipo de tres personas para crear **su propia versión de las Torres de Hanoi en arte ASCII**.

### Lo que deben entregar

1. **Un archivo `.cpp`** (o el cuaderno Jupyter) con:
   - Función para dibujar las torres en el estado dado
   - Función para mover un disco con validación
   - Una interfaz de consola simple (el usuario escribe origen y destino)

2. **Fotos o capturas de pantalla** mostrando las torres en al menos **4 estados diferentes**:
   - Estado inicial (todos en torre A)
   - Estado intermedio (los discos distribuidos)
   - Estado casi final
   - Estado final (todos en torre C)

### Criterios de diseño

Pueden personalizar **todo lo visual** — el código de lógica (`moverDisco`, las validaciones) debe funcionar correctamente, pero el arte es completamente libre:

| Aspecto | Ejemplo mínimo | Ideas para destacar |
|---|---|---|
| Forma de los discos | `===` | `###`, `░░░`, `▓▓▓`, `<=>`, `[---]` |
| Estilo del poste | `\|` | `║`, `I`, `!`, doble barra |
| Base de la torre | `===` | `▄▄▄`, `╔═══╗`, sombra con `_` |
| Separador entre torres | espacio | `│`, `  ‖  `, columnas decoradas |
| Etiquetas | `A B C` | nombres, emojis, numeración |
| Número de discos | 3–4 | hasta 6 si el diseño lo permite |

### Observaciones importantes

> **No resuelvan Hanoi todavía.**  
> El algoritmo recursivo que mueve los discos automáticamente es el tema de la sesión 13.  
> Hoy, los movimientos los hace el **usuario** escribiendo origen y destino.

> **El código que gane el concurso de la sesión 13** se usará como base para implementar el algoritmo recursivo.  
> Vale la pena que el código esté bien estructurado y sea legible, no solo bonito visualmente.

### Esqueleto para su entrega

```cpp
#include <iostream>
#include <vector>
#include <string>
using namespace std;

// ── Sus constantes de diseño ──────────────────────────────────────────────────
const int N_DISCOS  = 4;
const int ANCHO     = 11;  // pueden cambiar esto

// ── Sus funciones de dibujo ───────────────────────────────────────────────────
void dibujarFila(int disco)   { /* su diseño aquí */ }
void dibujarTorres(const vector<int>& A,
                   const vector<int>& B,
                   const vector<int>& C) { /* su diseño aquí */ }

// ── Lógica de movimiento (pueden mejorarla) ───────────────────────────────────
bool moverDisco(vector<int>& origen, vector<int>& destino) { /* ... */ }

// ── Interfaz de usuario ───────────────────────────────────────────────────────
int main() {
    vector<int> A(N_DISCOS), B, C;
    // inicializar A con discos N_DISCOS, N_DISCOS-1, ..., 1
    for (int i = N_DISCOS; i >= 1; i--) A.push_back(i);

    char origen, destino;
    while (/* condición de victoria */) {
        dibujarTorres(A, B, C);
        cout << "Mover de: "; cin >> origen;
        cout << "Mover a:  "; cin >> destino;
        // llamar a moverDisco con las torres correctas
    }
    cout << "¡Felicidades! Resolviste las Torres de Hanoi." << endl;
    return 0;
}
```


---

## Resumen

| Concepto | Símbolo/función | Qué hace |
|---|---|---|
| Fila vacía | `dibujarFila(0)` | Imprime el poste centrado sin disco |
| Fila con disco | `dibujarFila(k)` | Imprime barra de `2k-1` `=` centrada |
| Estado de torre | `vector<int>` | Índice 0 = fondo, último = tope |
| Dibujar estado | `dibujarTorres(A,B,C)` | Itera filas de arriba hacia abajo |
| Mover disco | `moverDisco(orig, dest)` | Valida regla de Hanoi y actualiza vectores |
| Arte ASCII | caracteres `\|=*#░▓` | Imágenes en texto plano |

---
*Programación Avanzada — Universidad Panamericana*